# 01 — Perch v2 Embedding Extraction over Unlabelled Soundscapes

Extracts Perch v2 embeddings and mapped 234-class logits for **all 10,658 BirdCLEF+ 2026 training soundscapes** (~127,896 five-second windows). The public embedding cache covered only the 59 labelled files; this notebook extends it to the full unlabelled set, producing the in-domain cache used to train the custom head in notebook `02`.

**Author:** Edidiong-Abasi Anwanane. **Own work.**

**Outputs (not committed):** `emb_all.npy` (n,1536), `scores_all.npy` (n,234), `meta_all.parquet`. These are derivatives of CC BY-NC-SA competition data and are reserved for follow-up work — see repo README.

**Inputs:** BirdCLEF+ 2026 competition data; `tuckerarrants/perch-v2-no-dft-onnx`; Perch `labels.csv`. Run on Kaggle (T4), ~1–2 h.

In [ ]:
# =============================================================================
# BirdCLEF+ 2026 — Path Y Day 1.5: Compute Perch v2 embeddings on
# all 10,658 train_soundscapes for pseudo-labeling
# =============================================================================
# What this does:
#   1. Loads ONNX Perch v2 (same model used by exp019).
#   2. Builds taxonomy mapping (same as exp019 — PRIMARY_LABELS, BC_INDICES,
#      MAPPED_POS / MAPPED_BC_IDX, proxy_map, NO_LABEL) so output is shape-
#      compatible with perch_meta / exp019's run_perch().
#   3. Smoke-tests on 5 files first — abort if outputs don't look sane.
#   4. Processes ALL 10,658 train_soundscapes in batches with checkpointing
#      every 500 files. If the kernel dies, restart this same notebook and
#      it resumes from the last checkpoint.
#   5. Final output: 3 arrays saved to /kaggle/working/
#        - emb_all.npy        (n_windows, 1536)   float32
#        - scores_all.npy     (n_windows, 234)    float32   (mapped + proxy)
#        - meta_all.parquet                       columns: row_id, filename, site, hour_utc
#      Where n_windows ≈ 10,658 × 12 ≈ 127,896.
#
# Setup BEFORE running:
#   • Accelerator: GPU T4 x2 (we use one)
#   • Internet: ON (for any minor pip needs — none expected)
#   • Inputs attached:
#       - BirdCLEF+ 2026 (competition data)                          REQUIRED
#       - jaejohn/perch-meta                                          (not used here,
#                                                                      but keep attached
#                                                                      for taxonomy file)
#       - tuckerarrants/perch-v2-no-dft-onnx                          REQUIRED (Perch ONNX + wheel)
#       - google/bird-vocalization-classifier (Kaggle Models)         REQUIRED (Perch labels.csv)
#         OR equivalent path that contains labels.csv with
#         "inat2024_fsd50k" or "scientific_name" column.
#
# Estimated runtime: ~1-2 hours on T4 (single GPU/CPU pass through ONNX).
# Memory usage: ~800 MB peak.
# =============================================================================

# =============================================================================
# CELL 1: Install ONNX runtime from wheel (no internet needed)
# =============================================================================
import subprocess, sys, os, glob
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

# Locate the ONNX runtime wheel
def _find_wheel(pattern):
    hits = sorted(INPUT_ROOT.rglob(pattern))
    return hits[0] if hits else None

ort_whl = _find_wheel("onnxruntime-1.24.4-*.whl")
if ort_whl:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ort_whl)])
    print(f"✓ Installed ONNX runtime from {ort_whl.name}")
else:
    # Last resort: try regular pip (only works if Internet is ON)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    print("✓ Installed ONNX runtime from PyPI")

import onnxruntime as ort
print(f"✓ ONNX Runtime: {ort.__version__}")

In [ ]:
# =============================================================================
# CELL 2: Imports & config (mirror exp019 exactly)
# =============================================================================
import os, sys, re, gc, time, json, math, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
import concurrent.futures

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)
CKPT_DIR = WORK / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# Audio constants — MUST match exp019
SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC      # 160_000
FILE_SAMPLES   = 60 * SR              # 1_920_000
N_WINDOWS      = 12                   # 12 × 5s = 60s

# Output paths
OUT_EMB    = WORK / "emb_all.npy"
OUT_SCORES = WORK / "scores_all.npy"
OUT_META   = WORK / "meta_all.parquet"

# Smoke test settings
SMOKE_N_FILES = 5

# Full run settings
BATCH_FILES = 16            # files per Perch forward pass
CHECKPOINT_EVERY = 500      # files between checkpoint saves

print(f"OUT_EMB    = {OUT_EMB}")
print(f"OUT_SCORES = {OUT_SCORES}")
print(f"OUT_META   = {OUT_META}")

In [ ]:
# =============================================================================
# CELL 3: Locate competition data + Perch ONNX
# =============================================================================
COMP_DIR_CANDIDATES = [
    "/kaggle/input/competitions/birdclef-2026",
    "/kaggle/input/birdclef-2026",
]
BASE = next((Path(p) for p in COMP_DIR_CANDIDATES if os.path.exists(p)), None)
assert BASE is not None, "Competition data not found."
print(f"✓ BASE = {BASE}")

ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                   next(INPUT_ROOT.glob("**/perch_v2*.onnx"), None))
assert ONNX_PERCH_PATH is not None, "Perch ONNX not found. Attach tuckerarrants/perch-v2-no-dft-onnx"
print(f"✓ ONNX_PERCH_PATH = {ONNX_PERCH_PATH}")

train_ss_dir = BASE / "train_soundscapes"
all_paths = sorted(train_ss_dir.glob("*.ogg"))
print(f"✓ Total train_soundscapes files: {len(all_paths)}")
assert len(all_paths) > 0, "No .ogg files in train_soundscapes"

In [ ]:
# =============================================================================
# CELL 4: Build taxonomy mapping (mirror exp019 L177-328)
# =============================================================================
taxonomy   = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
print(f"N_CLASSES = {N_CLASSES}")
assert N_CLASSES == 234, f"Expected 234 classes, got {N_CLASSES}"

# Locate Perch labels.csv (in a Perch model bundle or as separate attachment)
def _find_perch_labels_path():
    # Common locations
    candidates = []
    for p in sorted(Path("/kaggle/input").rglob("labels.csv")):
        try:
            cols = set(pd.read_csv(p, nrows=0).columns)
        except Exception:
            continue
        if {"inat2024_fsd50k", "scientific_name"} & cols:
            candidates.append(p)
    if candidates:
        print(f"✓ Found Perch labels at: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        "Perch labels.csv not found in /kaggle/input/. Attach the Perch "
        "model bundle (google/bird-vocalization-classifier on Kaggle Models)."
    )

bc_labels_path = _find_perch_labels_path()
bc_labels = pd.read_csv(bc_labels_path).reset_index().rename(columns={"index": "bc_index"})
if "scientific_name" not in bc_labels.columns:
    # alternative column name fallback
    for c in ["inat2024_fsd50k", "label", "labels", "name"]:
        if c in bc_labels.columns:
            bc_labels = bc_labels.rename(columns={c: "scientific_name"})
            break
assert "scientific_name" in bc_labels.columns, f"No usable label column in {bc_labels_path}"

NO_LABEL = len(bc_labels)
mapping = (taxonomy.merge(bc_labels, on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

# Proxy mapping for unmapped species (genus-level)
import re as _re
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

proxy_map = {}
unmapped_df = taxonomy[taxonomy["primary_label"].isin(
    [PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy()
for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci = str(row["scientific_name"])
    genus = sci.split()[0]
    hits = bc_labels[bc_labels["scientific_name"].astype(str)
                     .str.match(rf"^{_re.escape(genus)}\s", na=False)]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map = {idx: bcs for idx, bcs in proxy_map.items()
             if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA}
print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | "
      f"No signal: {len(UNMAPPED_POS) - len(proxy_map)}")

In [ ]:
# =============================================================================
# CELL 5: Setup ONNX session
# =============================================================================
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                     providers=["CPUExecutionProvider"])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print(f"✓ ONNX session ready. Input: {ONNX_INPUT_NAME}")
print(f"  Output map: {ONNX_OUT_MAP}")
assert "label" in ONNX_OUT_MAP and "embedding" in ONNX_OUT_MAP, \
    "Expected 'label' and 'embedding' outputs from Perch ONNX"

In [ ]:
# =============================================================================
# CELL 6: Helpers (mirror exp019 read_60s + parse_fname + run_perch)
# =============================================================================
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y

def run_perch_batch(paths, batch_files=16, verbose=False):
    """Mirror of exp019's run_perch — returns (meta_df, scores, embs).
    Identical logic so outputs are interchangeable with the original."""
    paths = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536), dtype=np.float32)
    wr = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch", disable=not verbose)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_exec:
        next_paths = paths[0:batch_files]
        future_audio = [io_exec.submit(read_60s, p) for p in next_paths]
        for start in itr:
            batch_paths = next_paths
            batch_n     = len(batch_paths)
            batch_audio = [f.result() for f in future_audio]
            ns = start + batch_files
            if ns < len(paths):
                next_paths   = paths[ns:ns + batch_files]
                future_audio = [io_exec.submit(read_60s, p) for p in next_paths]
            x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr
            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr+N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr+N_WINDOWS] = path.name
                sites    [wr:wr+N_WINDOWS] = meta["site"]
                hours    [wr:wr+N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS
            outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
            emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb
            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

In [ ]:
# =============================================================================
# CELL 7: SMOKE TEST on 5 files — verify outputs look like perch_meta's
# =============================================================================
print("=" * 70)
print(f"SMOKE TEST: processing {SMOKE_N_FILES} files")
print("=" * 70)
t0 = time.time()
smoke_meta, smoke_scores, smoke_embs = run_perch_batch(
    all_paths[:SMOKE_N_FILES], batch_files=BATCH_FILES, verbose=True
)
print(f"Smoke test time: {time.time() - t0:.1f}s")
print(f"\n  Output shapes:")
print(f"    meta: {smoke_meta.shape}, columns: {list(smoke_meta.columns)}")
print(f"    scores: {smoke_scores.shape}, dtype: {smoke_scores.dtype}")
print(f"    embs: {smoke_embs.shape}, dtype: {smoke_embs.dtype}")
print(f"\n  Embedding stats: mean={smoke_embs.mean():.4f}, std={smoke_embs.std():.4f}, "
      f"min={smoke_embs.min():.4f}, max={smoke_embs.max():.4f}")
print(f"  Scores stats:   mean={smoke_scores.mean():.4f}, "
      f"max={smoke_scores.max():.4f}, nonzero frac={(smoke_scores > 0).mean():.4f}")
print(f"\n  First 3 rows of meta:")
print(smoke_meta.head(3).to_string())

# Assertions
expected_rows = SMOKE_N_FILES * N_WINDOWS
assert smoke_embs.shape == (expected_rows, 1536), f"emb shape wrong: {smoke_embs.shape}"
assert smoke_scores.shape == (expected_rows, N_CLASSES), f"scores shape wrong: {smoke_scores.shape}"
assert len(smoke_meta) == expected_rows
assert smoke_embs.std() > 1e-4, "Embeddings look constant/zero — Perch may be broken"
assert smoke_scores.max() > 0, "All scores are zero — class mapping may be broken"
print("\n✓ SMOKE TEST PASSED")

# Estimate full runtime
files_per_sec = SMOKE_N_FILES / (time.time() - t0)
est_full_s = len(all_paths) / files_per_sec
print(f"\n  → Estimated full-run time: {est_full_s/60:.0f} min "
      f"for {len(all_paths)} files")
if est_full_s > 8 * 3600:
    print(f"  ⚠ WARNING: estimated > 8 hours. Consider raising BATCH_FILES.")

In [ ]:
# =============================================================================
# CELL 8: FULL RUN with checkpointing
# =============================================================================
print("=" * 70)
print(f"FULL RUN: {len(all_paths)} files with checkpointing every "
      f"{CHECKPOINT_EVERY} files")
print("=" * 70)

# Find resume point: scan checkpoints/ for highest finished index
def find_resume_point():
    saved = sorted(CKPT_DIR.glob("checkpoint_*.npz"),
                   key=lambda p: int(p.stem.split("_")[-1]))
    if not saved:
        return 0, [], [], []
    last = saved[-1]
    n_done = int(last.stem.split("_")[-1])
    print(f"  → Resuming from checkpoint at file {n_done}")
    data = np.load(last, allow_pickle=True)
    return n_done, [data["embs"]], [data["scores"]], [data["meta_records"]]

start_idx, embs_chunks, scores_chunks, meta_chunks = find_resume_point()
remaining_paths = all_paths[start_idx:]
print(f"  Files to process: {len(remaining_paths)} (skipping first {start_idx})")

t_full_start = time.time()
processed = start_idx
chunk_size = CHECKPOINT_EVERY

for chunk_start in range(0, len(remaining_paths), chunk_size):
    chunk_paths = remaining_paths[chunk_start:chunk_start + chunk_size]
    t0 = time.time()
    meta_df, scores, embs = run_perch_batch(chunk_paths, batch_files=BATCH_FILES,
                                              verbose=False)
    dt = time.time() - t0
    processed += len(chunk_paths)

    embs_chunks.append(embs)
    scores_chunks.append(scores)
    meta_chunks.append(meta_df.to_records(index=False))

    # Save checkpoint
    ckpt_path = CKPT_DIR / f"checkpoint_{processed:05d}.npz"
    np.savez_compressed(ckpt_path,
                         embs=embs,
                         scores=scores,
                         meta_records=meta_df.to_records(index=False))
    # Keep only the most recent 2 checkpoints to save disk
    old_ckpts = sorted(CKPT_DIR.glob("checkpoint_*.npz"),
                        key=lambda p: int(p.stem.split("_")[-1]))
    for old in old_ckpts[:-2]:
        old.unlink()

    elapsed = time.time() - t_full_start
    rate = len(chunk_paths) / dt
    eta = (len(all_paths) - processed) / rate
    print(f"  {processed}/{len(all_paths)} files | "
          f"chunk {dt:.0f}s ({rate:.1f} files/s) | "
          f"elapsed {elapsed/60:.1f}min | ETA {eta/60:.0f}min")

# Concatenate all
print("\n  Concatenating all chunks...")
all_embs   = np.concatenate(embs_chunks, axis=0)
all_scores = np.concatenate(scores_chunks, axis=0)
all_meta_recs = np.concatenate(meta_chunks, axis=0)
all_meta = pd.DataFrame.from_records(all_meta_recs)

print(f"  Final shapes:")
print(f"    embs:   {all_embs.shape}")
print(f"    scores: {all_scores.shape}")
print(f"    meta:   {all_meta.shape}")
print(f"  Total wall time: {(time.time() - t_full_start)/60:.1f} min")

In [ ]:
# =============================================================================
# CELL 9: Save final outputs to /kaggle/working/
# =============================================================================
print("=" * 70)
print("Saving final outputs")
print("=" * 70)

# Save as compressed npz for the big arrays (saves ~50% disk vs npy for floats)
final_npz_path = WORK / "perch_train_soundscapes_all.npz"
np.savez_compressed(final_npz_path,
                     embs=all_embs.astype(np.float32),
                     scores=all_scores.astype(np.float32))
print(f"✓ Saved arrays: {final_npz_path} "
      f"({final_npz_path.stat().st_size/1e6:.1f} MB)")

# Save metadata as parquet
final_meta_path = WORK / "perch_train_soundscapes_meta.parquet"
all_meta.to_parquet(final_meta_path, index=False)
print(f"✓ Saved metadata: {final_meta_path} "
      f"({final_meta_path.stat().st_size/1e6:.1f} MB)")

# Clean up checkpoints
for c in CKPT_DIR.glob("checkpoint_*.npz"):
    c.unlink()
CKPT_DIR.rmdir()
print("✓ Cleaned up checkpoint dir")

print("\n" + "=" * 70)
print("DONE. Next steps:")
print("=" * 70)
print(f"""
1. Click 'Save Version' → 'Save & Run All (Commit)' to lock the outputs.
2. Once committed, go to the notebook's 'Output' tab.
3. Click 'New Dataset' and include BOTH:
     - perch_train_soundscapes_all.npz
     - perch_train_soundscapes_meta.parquet
4. Name the dataset: birdclef-perch-soundscapes-embs-v01
5. Make it Private. Create.
6. This dataset is consumed by the head-training pipeline that
   uses Perch v2 embeddings as input features for a small MLP
   classifier on top.
""")